<a href="https://colab.research.google.com/github/saulo-albuquerque-phys/GWgpu-jax/blob/sharpy_sampler/examples/public_Parameter_Estimation_mlgw_bbh_GW150914_sharpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GWgpu_jax — **SHARPy** PE on **real GW150914** with **mlgw_bbh_jax** (SEOBNRv5HM, **(2,2) mode**, non-uniform priors, `mc`/`q` sampling)

This is the **SHARPy** counterpart of
`public_Parameter_Estimation_mlgw_bbh_GW150914_acceptance_walk.ipynb`. The data,
PSDs, waveform model, likelihood, parameterisation and priors are **identical**;
only the sampler changes.

**SHARPy** (*Sequential HAmiltonian Riemann montecarlo PYthon sampler*; Demasi,
Del Pozzo et al.) is an adaptively-tempered **Sequential Monte Carlo** sampler:

1. Start from `N` prior draws at inverse temperature `β = 0`.
2. Choose the next `β` by bisection so that the ESS of the incremental weights
   `L^{Δβ}` is `alpha · N` (`alpha = 0.95`).
3. Accumulate the evidence `log Z += log mean(L^{Δβ})`, then resample.
4. Move every particle with **NUTS** whose metric is the **SoftAbs-regularised
   Hessian** of the tempered log-posterior at that particle (Riemannian
   preconditioning), with reflective/periodic boundaries.
5. Repeat until `β = 1`. Posterior draws are rejection-sampled from the
   *recycled* mixture of all tempered populations.

It runs through `gwgpu_jax.GWgpu_jaxSHARPySampler`, a pure-JAX port that uses the
same BlackJAX gwgpu_jax already pins. You don't need the upstream SHARPy
BlackJAX fork or `netket`, and nothing extra has to be fetched.

Because SHARPy is **gradient-based**, every step evaluates gradients *and*
per-particle Hessians of the mlgw_bbh likelihood (all through JAX autodiff of
the tf2jax network). Each SMC step therefore costs more than a
nested-sampling step, but far fewer steps are needed.

| Parameter     | Prior                 | Density        |
|---------------|-----------------------|----------------|
| `mc`, `q`     | uniform               | —              |
| `chi_1`, `chi_2` | uniform            | —              |
| `inclination` | isotropic orientation | `p(θ) ∝ sin θ` |
| `dec`         | isotropic sky         | `p(δ) ∝ cos δ` |
| `distance`    | uniform in volume     | `p(d) ∝ d²`    |
| `ra`, `psi`, `phi_c`, `tc` | uniform  | — (`ra`, `psi`, `phi_c` periodic) |

**Runtime → Change runtime type → A100 GPU** (recommended). A T4 also works if you
lower `CHUNK_SIZE` (section 6) to avoid running out of GPU memory in the Hessian.

## 1. Install GWgpu_jax

[GWgpu-jax](https://github.com/saulo-albuquerque-phys/GWgpu-jax) is a **public**
repository — no GitHub token or authentication is needed. The cell below installs
it straight from GitHub.

Colab GPU runtimes already ship a CUDA-enabled JAX, so the `gpu` extra is not
required; the cell re-pins JAX to the 0.4.x line that Colab's CUDA plugin
supports. Requires **Python 3.11**, which is what Colab provides.


In [ ]:
import importlib.util, os

RUNNING_ON_COLAB  = importlib.util.find_spec("google.colab") is not None
ALREADY_INSTALLED = importlib.util.find_spec("gwgpu_jax") is not None

if RUNNING_ON_COLAB and not ALREADY_INSTALLED:
    # Pin JAX to the 0.4.x line Colab's CUDA plugin still supports.
    !pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
    !pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"

    # Public repo - no token, no authentication. Installs the `sharpy_sampler` branch,
    # which carries gwgpu_jax.GWgpu_jaxSHARPySampler.
    !pip install -q "gwgpu_jax[data,mlgw] @ git+https://github.com/saulo-albuquerque-phys/GWgpu-jax.git@sharpy_sampler"

    print("Installed. If JAX was re-pinned: Runtime -> Restart session, then re-run from here.")
else:
    print("gwgpu_jax already importable - skipping install.")

### 1b. Pull the vendored `mlgw_bbh_jax` sub-repo (Colab bandaid)

IMPORTANT: RESTART THE SECTION BEFORE RUNNING HERE. GO IN EXEC ENVIRONMENTS IN THE TOP BAR ON TOP OF THE NOTEBOOK, CLICK THERE, AND THEN CLICK RESTART SECTION, AND THEN YOU CAN START FROM THE CELL BELLOW AND GO DOWNWARDS!

`mlgw_bbh_jax` is shipped as a *vendored* sub-repo at
`<gwgpu_jax-install>/mlgw_jax/mlgw_bbh_jax/`. It is currently **not committed to
the main branch** of the GWgpu_jax repository that Colab clones from, so the
wheel built on Colab is missing it. The cell below clones it on the fly from
the **`gwgpu_jax_compatible`** branch of `adrianomascioli/MLGW-JAX`, which
already contains the inference-compatibility fixes — so, unlike the original
notebook, **no on-the-fly source patch is applied**.

Skip this cell if you're running locally against a working tree that
already has `mlgw_bbh_jax/` populated.

In [ ]:
# ── Vendored-subrepo bandaid for `gwgpu_jax[mlgw]` on Colab ────────────
# The `mlgw_bbh_jax` Python package is a vendored sub-repo that lives at
#     <gwgpu_jax-install>/mlgw_jax/mlgw_bbh_jax/
# but it's not committed to the GWgpu_jax repo Colab clones from, so the
# wheel built on Colab is missing it. We clone it from the upstream
# `gwgpu_jax_compatible` branch of adrianomascioli/MLGW-JAX, which already
# bakes in the inference-compatibility fixes (Keras 2/3 model-load compat,
# optional keras_tuner / GW_helper imports, JAX-traceable Wigner-d), so
# — unlike the original notebook — NO on-the-fly patch is applied.
#
# Once `mlgw_bbh_jax` is properly committed to the GWgpu_jax repo (or pinned
# as a submodule), this whole cell can be deleted.
import subprocess, sys, shutil
from pathlib import Path

# FIX: Downgrade cryptography to fix pyOpenSSL incompatibility (module 'lib' has no attribute 'GEN_EMAIL')
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "cryptography<44", "pyopenssl"],
    check=True,
)

# 1. Install training-only deps the upstream sub-repo may import. On the
#    gwgpu_jax_compatible branch these imports are already guarded, so this
#    is belt-and-braces and can be dropped if you like.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "keras-tuner"],
    check=True,
)

# 2. Clone the vendored sub-repo (gwgpu_jax_compatible branch) into the
#    gwgpu_jax install path.
import gwgpu_jax
_GW    = Path(gwgpu_jax.__file__).resolve().parent
_DEST  = _GW / "mlgw_jax" / "mlgw_bbh_jax"
_READY = _DEST / ".gwgpu_jax_compatible_ready"

if _READY.exists():
    print(f"✓ mlgw_bbh_jax (gwgpu_jax_compatible) already present at {_DEST}")
else:
    if _DEST.exists():
        shutil.rmtree(_DEST)
    print(f"Cloning mlgw_bbh_jax (gwgpu_jax_compatible) into {_DEST} (~1 GB, ~1–2 min on Colab)…")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", "gwgpu_jax_compatible",
        "https://github.com/adrianomascioli/MLGW-JAX.git",
        str(_DEST),
    ], check=True)
    _READY.touch()
    print(f"✓ mlgw_bbh_jax cloned (gwgpu_jax_compatible) at {_DEST}")

# 3. Verify the wrapper can now import the vendored Python package.
from gwgpu_jax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator  # noqa: F401
print("✓ MLGWBBHGenerator importable")

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # silence TF startup spam

import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Enable double precision — important for likelihood accuracy.
jax.config.update("jax_enable_x64", True)

import gwgpu_jax
print("gwgpu_jax version :", gwgpu_jax.__version__)
print("JAX devices   :", jax.devices())

## 2. Build the detector network (GW150914 standard window)

GW150914 was observed by H1 and L1. We use the LVC-standard analysis
window for this event: **4 s @ 4 kHz** in the **20–1024 Hz** band. The
`TimeFrequencyGrid` is shared across all detectors in the network.

In [ ]:
DURATION       = 4.0
SAMPLING_RATE  = 4096.0
F_MIN, F_MAX   = 20.0, 1024.0

grid = gwgpu_jax.TimeFrequencyGrid(
    duration=DURATION, sampling_rate=SAMPLING_RATE,
    f_min=F_MIN, f_max=F_MAX,
)
network = gwgpu_jax.Network.from_names(["H1", "L1"], grid)
print(grid)
print(network)

## 3. Fetch real GW150914 strain from GWOSC

`gwgpu_jax.compat.attach_event_to_network` does three things in one shot:

1. Downloads H1 and L1 strain around the GW150914 trigger via
   `gwpy.TimeSeries.fetch_open_data` (needs an internet connection).
2. Estimates each IFO's PSD from an **adjacent off-source** segment (32 s
   ending 8 s before the trigger) with Welch's method.
3. Crops + resamples the on-source segment so the **merger sits at
   0.875 × duration = 3.5 s** into the segment, then attaches strain
   and PSD to every IFO in the network.

The 3.5 s offset is the convention the GWgpu_jax `tc` parameter must match
(see cell 11).

In [ ]:
import time

t0 = time.perf_counter()
gwgpu_jax.compat.attach_event_to_network(
    network, "GW150914",
    estimate_psd          = True,
    psd_segment_duration  = 32.0,
    psd_offset            = 8.0,
    verbose               = False,
)
print(f"GW150914 strain + PSDs attached in {time.perf_counter()-t0:.1f} s")
print(f"  populated detectors : {[ifo.name for ifo in network.interferometers]}")

## 4. Build a `(params, freqs) -> (h₊, hₓ)` waveform_fn from mlgw_bbh_jax

`MLGWBBHGenerator` is a **time-domain** waveform model: it returns h₊(t),
hₓ(t) on a user-supplied time grid with the merger (waveform peak) placed
at **t = 0**. The GWgpu_jax sampler expects an **FD** waveform
`(params, freqs) -> (ĥ₊, ĥₓ)`, so the shim below converts TD → FD with the
GWgpu_jax convention `ĥ(f_k) = rfft(h)[k] × dt`. Two details (both from the
upstream `mlgw.GW_FD_generator`) are **essential for real-data PE** and are
invisible to inject↔recover, so they're easy to miss:

* **The time grid must cross zero.** mlgw_bbh zeros the amplitude outside its
  native range, so a grid that stops at `-dt` drops the peak *and the whole
  ringdown*, leaving a distorted template that rails `tc` and the masses at
  the prior edges. We evaluate on `[final_time − duration, final_time − dt]`
  with `final_time = 0.01 s`, so the peak and ringdown are sampled.
* **Taper before the rFFT.** An asymmetric Tukey window (long `alpha_left`
  on the inspiral turn-on, tiny `alpha_right` on the ringdown edge)
  suppresses the spectral leakage from the hard segment boundaries.

Two notes about the parameter mapping:

* We sample the **chirp mass `mc` and mass ratio `q = m₂/m₁ ≤ 1`** and
  convert to component masses inside the waveform_fn via
  `m₁ = mc·(1+q)^{1/5}·q^{-3/5}`, `m₂ = q·m₁`.
  `mlgw_bbh_jax` requires **m₁ ≥ m₂**, which `q ≤ 1` guarantees by
  construction — so no internal mass/spin swap and no label-swap
  degeneracy. The posterior is directly `(mc, q, χ₁, χ₂)` with `χ₁` the
  heavier-body spin.
* `phi_c` in our particle dict feeds the surrogate's reference orbital
  phase `phi_0` and the projection-layer phase.

The model loads (TensorFlow → tf2jax) on first construction; the JIT path is
cached internally so repeated calls with the same time grid don't recompile.

In [ ]:
from gwgpu_jax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator

# ── TD→FD recipe (self-contained so it works against any installed gwgpu_jax) ─────
# Two corrections vs the naive `rfft(hp) * dt`, both lifted from the upstream
# `mlgw.GW_FD_generator` and both essential for *real-data* PE (they cancel in
# inject↔recover, so synthetic tests don't catch them):
#
#   1. The mlgw time grid must CROSS ZERO. mlgw_bbh puts the waveform peak at
#      t = 0 and zeros the amplitude outside its native range. A grid that
#      stops at -dt (the old `time_domain_array_mlgw`) drops the peak + the
#      whole ringdown -> distorted template that rails tc and the masses.
#      We rebuild it as [final_time - duration, final_time - dt] with a small
#      positive `final_time` so the peak + ringdown are sampled.
#   2. An asymmetric Tukey taper applied in the time domain before the rFFT,
#      to kill spectral leakage from the inspiral turn-on (long alpha_left)
#      and the truncated ringdown edge (tiny alpha_right).

MLGW_FINAL_TIME = 0.01   # s of post-merger time kept at the tail (peak+ringdown)
TUKEY_ALPHA_LEFT, TUKEY_ALPHA_RIGHT = 0.2, 0.001

def _tukey_asymmetric(n, alpha_left, alpha_right):
    """Asymmetric Tukey window (JAX-traceable). Matches GW_FD_generator."""
    x = jnp.linspace(0.0, 1.0, n)
    w = jnp.ones_like(x)
    left  = 0.5 * (1.0 + jnp.cos(2.0 * jnp.pi * (x / alpha_left - 0.5)))
    right = 0.5 * (1.0 + jnp.cos(2.0 * jnp.pi * ((x - 1.0) / alpha_right + 0.5)))
    w = jnp.where(x < alpha_left  / 2.0, left,  w)
    w = jnp.where(x > 1.0 - alpha_right / 2.0, right, w)
    return w

def build_mlgw_bbh_waveform_fn(grid, modes=None,
                               final_time=MLGW_FINAL_TIME,
                               alpha_left=TUKEY_ALPHA_LEFT,
                               alpha_right=TUKEY_ALPHA_RIGHT):
    """Return a JAX-traceable ``(params, freqs) -> (hp, hc)`` callable.

    The TD surrogate is evaluated on a zero-crossing grid and converted to the
    GWgpu_jax FD convention ĥ(f_k) = rfft(h)[k]·dt after an asymmetric Tukey taper.

    Either mass ordering is fine: mlgw_bbh swaps m1 <-> m2 (with their spins)
    internally, so the surrogate is always evaluated in its q_mlgw = m1/m2 in
    [1, 10] domain.

    NOTE on ``model=4``: the wrapper default is the string ``"model_4"``, but
    ``mlgw.GW_generator`` only resolves the bundled TD_models/model_<N> path
    for an **int**; passing ``model=4`` finds the SEOBNRv5HM weights.
    """
    bbh    = MLGWBBHGenerator(model=4)
    dt     = grid.dt
    N      = grid.n_samples
    # Zero-crossing grid: [final_time - duration, final_time - dt], step dt.
    # (Equivalent to grid.time_domain_array_mlgw once the package carries the
    #  mlgw_final_time fix; rebuilt here so the notebook is version-independent.)
    t_grid = jnp.arange(N) * dt - grid.duration + final_time
    window = _tukey_asymmetric(N, alpha_left, alpha_right)
    gen_wf = bbh.generate  # = self._gen.get_WF (pure JAX, traceable)

    def waveform_fn(params, freqs):
        # Sample the chirp mass `mc` and mass ratio `q = m2/m1 in (0, 1]`
        # directly, then convert to component masses. q <= 1 enforces m1 >= m2
        # by construction, so the surrogate is always called in its m1 >= m2
        # domain and there is NO label-swap degeneracy (no heavier-first
        # relabelling needed downstream).
        mc     = params["mc"]
        q      = params["q"]                       # q = m2/m1 in (0, 1]
        m1     = mc * (1.0 + q) ** (1.0 / 5.0) / q ** (3.0 / 5.0)
        m2     = q * m1
        chi_1  = params["chi_1"]
        chi_2  = params["chi_2"]

        # theta = [m1, m2, s1z, s2z, d_L, inclination, phi_0]
        theta = jnp.stack([
            m1, m2,
            chi_1, chi_2,
            params["distance"],
            params["inclination"],
            params["phi_c"],
        ])
        hp_td, hc_td = gen_wf(theta, t_grid, modes=modes)
        # Tukey taper in TD, then rFFT * dt (GWgpu_jax FD convention).
        hp_fd = jnp.fft.rfft(hp_td * window) * dt
        hc_fd = jnp.fft.rfft(hc_td * window) * dt
        return hp_fd, -hc_fd

    return waveform_fn

t0 = time.perf_counter()
waveform_fn = build_mlgw_bbh_waveform_fn(grid, modes=(2, 2))
print(f"mlgw_bbh_jax (SEOBNRv5HM, model_4, (2,2) mode) loaded in {time.perf_counter()-t0:.1f} s")


In [ ]:
# Check that the sampler is present in the installed gwgpu_jax (fails loudly if
# Colab installed an older branch).
from gwgpu_jax import GWgpu_jaxSHARPySampler
print("✓ GWgpu_jaxSHARPySampler available")

## 5. Set up the SHARPy sampler (with non-uniform priors)

The 11-D prior is **exactly** the one used by the acceptance-walk notebook:
`PARAM_BOUNDS` sets each parameter's support, and `PRIORS` overrides the shape
of the geometric ones. SHARPy samples on the unit hypercube, so the gwgpu_jax
prior inverse-CDFs map each particle to physical space. The prior is uniform
on the cube, which is SHARPy's native set-up.

Boundary conditions follow SHARPy: **`ra`, `phi_c`, `psi` are periodic** (the
antenna patterns have period π in `psi`). All other coordinates are
**reflective**: a NUTS trajectory that hits a wall bounces back with its
momentum flipped.

In [ ]:
TC_CENTER = 0.875 * grid.duration   # 3.5 s for the 4-s GW150914 segment

# Sample the chirp mass `mc` and mass ratio `q = m2/m1 in (0, 1]` directly
# (instead of the component masses). The waveform_fn converts (mc, q) ->
# (m1, m2); because q <= 1 enforces m1 >= m2, there is no label-swap degeneracy
# and no heavier-first relabelling is needed. Bounds are GW150914-centred
# (mc ~ 28.7, q ~ 0.86); q <= 1 maps to the surrogate's q_mlgw = m1/m2 = 1/q in
# [1, 2], well inside its [1, 10] validity range. The tc prior is +-0.1 s around
# the merger.
PARAM_BOUNDS = {
    "mc":          (26.0, 32.0),
    "q":           (0.6,  1.0),
    "chi_1":       (-0.4, 0.4),
    "chi_2":       (-0.4, 0.4),
    "distance":    (150.0, 700.0),
    "inclination": (0,  float(jnp.pi)),
    "ra":          (0.0,  2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0,  float(jnp.pi)),
    "phi_c":       (0.0,  2.0 * float(jnp.pi)),
    "tc":          (TC_CENTER - 0.1, TC_CENTER + 0.1),
}
FIXED_PARAMS = {}

# ── NEW: non-uniform priors on the geometric parameters ─────────────────
# String aliases resolved by gwgpu_jax.gwgpu_jax_prior_definitions over the
# matching PARAM_BOUNDS support. Equivalent explicit form:
#   from gwgpu_jax import SinUniform, CosUniform, Volumetric
#   PRIORS = {
#       "inclination": SinUniform(0.0, float(jnp.pi)),
#       "dec":         CosUniform(-float(jnp.pi)/2, float(jnp.pi)/2),
#       "distance":    Volumetric(100.0, 800.0),
#   }
PRIORS = {
    "inclination": "sin",          # p(θ) ∝ sin θ   (isotropic orientation)
    "dec":         "cos",          # p(δ) ∝ cos δ   (isotropic sky)
    "distance":    "volumetric",   # p(d) ∝ d²      (uniform in volume)
}

# Published-median reference (GWTC-1) for the corner-plot overlay only.
PUBLISHED_MEDIAN = dict(
    mc=28.71, q=0.860,  # GW150914 GWTC-1 detector-frame (mc, q=m2/m1) from m1=35.6, m2=30.6
    chi_1=0.0, chi_2=0.0,
    distance=440.0, inclination=2.5,
    ra=2.21, dec=-1.25, psi=0.0,
    phi_c=0.0,
    tc=TC_CENTER,
)

# SHARPy: tempered SMC with SoftAbs-Hessian NUTS moves. Same likelihood and
# priors as the nested-sampling runs; only the sampling algorithm differs.
sampler = gwgpu_jax.GWgpu_jaxSHARPySampler(
    network         = network,
    waveform_fn     = waveform_fn,
    param_bounds    = PARAM_BOUNDS,
    fixed_params    = FIXED_PARAMS,
    priors          = PRIORS,
    gmst            = None,                         # auto-uses network.gmst
    periodic_params = ("ra", "phi_c", "psi"),       # SHARPy's periodic angles
)
print(f"sampling dimension : {len(sampler.param_bounds)}")
print(f"free parameters    : {list(sampler.param_bounds)}")
print(f"periodic           : {sampler.periodic_params}")
print("priors             :", {k: type(v).__name__ for k, v in sampler._prior_specs.items()})
print(f"fixed parameters   : {FIXED_PARAMS}")

### 5b. Sanity check: the likelihood is differentiable

SHARPy needs `∇ log L` and `∇² log L`. The cell below evaluates both at the
centre of the prior. Every entry should be finite. It also gives an early
estimate of the per-particle Hessian cost.

In [ ]:
_u0      = jnp.full(len(sampler.param_names), 0.5)
_loglik  = sampler._unit_cube_loglik()

t0 = time.perf_counter()
_g = jax.jit(jax.grad(_loglik))(_u0).block_until_ready()
print(f"grad    finite: {bool(jnp.all(jnp.isfinite(_g)))}   (compile+run {time.perf_counter()-t0:.1f} s)")
t0 = time.perf_counter()
_H = jax.jit(jax.hessian(_loglik))(_u0).block_until_ready()
print(f"hessian finite: {bool(jnp.all(jnp.isfinite(_H)))}   (compile+run {time.perf_counter()-t0:.1f} s)")

## 6. Run SHARPy

Knobs (defaults = upstream SHARPy):

- `NUM_PARTICLES`: SMC population size. The original SHARPy GW150914 runs used
  2k–16k. More particles give a smoother posterior and a smaller `logZ` error.
- `ALPHA = 0.95`: target ESS fraction per tempering step. Values closer to 1
  give more, smaller temperature steps.
- `STEP_SIZE = 0.3`, `MAX_NUM_DOUBLINGS = 6`: NUTS leapfrog step size (fixed,
  because the Hessian metric absorbs the scale) and tree-depth cap
  (≤ 64 leapfrog steps).
- `NUM_MCMC_STEPS = 1`: NUTS moves per particle per SMC step. Raise it to 2–3 if
  the posterior shows many duplicated particles or the acceptance is low.
- `METRIC = "hessian"`: per-particle SoftAbs Hessian (SHARPy). `"ensemble"` uses
  the population covariance instead. It is cheaper (no Hessians) and leaves each
  tempered target exactly invariant, which makes it a useful cross-check.
- `CHUNK_SIZE`: particles per vectorised batch. **Lower it (e.g. 100–200) if you
  hit a GPU out-of-memory error**; it does not change the result.

**Compilation warning.** The first SMC step compiles the likelihood, its
Hessian and the NUTS kernel through the tf2jax SEOBNRv5HM network. Expect a
few minutes before the first progress line. Later steps reuse the compiled
program, because `β` is a traced argument.

Results are also saved to `OUTDIR/<label>_result.npz`.

In [ ]:
NUM_PARTICLES     = 4000
ALPHA             = 0.95
STEP_SIZE         = 0.3
MAX_NUM_DOUBLINGS = 6
NUM_MCMC_STEPS    = 1
METRIC            = "hessian"     # or "ensemble"
CHUNK_SIZE        = 500           # lower on OOM (T4: ~100–200)
OUTDIR            = "/content/sharpy_GW150914" if RUNNING_ON_COLAB else "sharpy_GW150914"

t0 = time.perf_counter()
result = sampler.run(
    rng_key           = jax.random.PRNGKey(43),
    num_particles     = NUM_PARTICLES,
    alpha             = ALPHA,
    step_size         = STEP_SIZE,
    max_num_doublings = MAX_NUM_DOUBLINGS,
    num_mcmc_steps    = NUM_MCMC_STEPS,
    metric            = METRIC,
    chunk_size        = CHUNK_SIZE,
    outdir            = OUTDIR,
    label             = f"mlgw_bbh_GW150914_sharpy_N{NUM_PARTICLES}",
    verbose           = True,
)
elapsed = time.perf_counter() - t0

n_post = len(next(iter(result.posterior_samples.values())))
print(f"\ntotal wall-clock   = {elapsed:.1f} s")
print(f"  SMC steps        = {result.n_steps}")
print(f"  log Z            = {result.logZ:+.3f} ± {result.logZ_err:.3f}")
print(f"  posterior draws  = {n_post} (recycled)  +  {NUM_PARTICLES} final particles")

In [ ]:
minutes = elapsed / 60
print(f"Total time for PE: {minutes:.2f} minutes")

### 6b. SMC diagnostics

* **β schedule**: should rise smoothly from ~0 to 1. A long run of tiny steps
  means the likelihood is very peaked relative to the prior, which is normal
  for a loud event.
* **NUTS acceptance**: around 0.6–0.95 is healthy. A sustained drop or many
  divergences suggests a smaller `STEP_SIZE` or more `NUM_MCMC_STEPS`.
* **Unique particles** at `β = 1`: after resampling many particles are
  duplicates. A low unique fraction means the mutation step isn't
  decorrelating enough; raise `NUM_MCMC_STEPS`.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
steps = np.arange(1, result.n_steps + 1)
axs[0].semilogy(steps, result.betas, "o-", ms=3); axs[0].set(xlabel="SMC step", ylabel=r"$\beta$", title="tempering schedule")
axs[1].plot(steps, result.acceptance, "o-", ms=3, label="acceptance")
axs[1].plot(steps, result.divergences, "o-", ms=3, label="divergent fraction")
axs[1].set(xlabel="SMC step", ylim=(0, 1.05), title="NUTS mutation"); axs[1].legend()
axs[2].plot(steps, result.ess / NUM_PARTICLES, "o-", ms=3); axs[2].axhline(ALPHA, color="k", ls=":")
axs[2].set(xlabel="SMC step", ylabel="ESS / N", ylim=(0, 1.05), title="incremental ESS")
plt.tight_layout(); plt.show()

U_final = result.history["particles"][-1]
n_unique = len(np.unique(np.round(U_final, 12), axis=0))
print(f"unique particles at β=1 : {n_unique}/{NUM_PARTICLES} ({100*n_unique/NUM_PARTICLES:.1f} %)")

## 7. Posterior summary

The "reference" column is the GWTC-1 published median. It is a visual aid only,
not a truth target. Summaries use the **recycled** posterior draws (SHARPy's
default). `result.final_samples` holds the `β = 1` population, for comparison.

In [ ]:
# No heavier-first relabelling needed: we sampled q = m1/m2 >= 1 directly, so
# m1 >= m2 holds by construction and there is no (m1,chi_1) <-> (m2,chi_2)
# label-swap degeneracy to collapse. chi_1 is already the heavier-body spin.

print(f"  {'param':12s}  {'median':>10s}  {'-1σ':>8s}  {'+1σ':>8s}  reference")
degenerate = []
for name in sampler.param_bounds:
    s = jnp.asarray(result.posterior_samples[name])
    lo, mid, hi = jnp.percentile(s, jnp.array([16.0, 50.0, 84.0]))
    if (hi - lo) < 1e-12 * jnp.maximum(jnp.abs(mid), 1.0):
        degenerate.append(name)
    ref_str = f"{PUBLISHED_MEDIAN[name]:+.3f}"
    print(f"  {name:12s}  {float(mid):+10.3f}  {float(mid-lo):8.3f}  {float(hi-mid):8.3f}  {ref_str}")

if degenerate:
    print(
        f"\n⚠  posterior columns collapsed (no spread): {degenerate}\n"
        f"   SMC steps={result.n_steps}  →  SHARPy likely under-mixed.\n"
        f"   Raise NUM_PARTICLES / NUM_MCMC_STEPS, then re-run the sampler cell."
    )

## 8. Corner plot

Red: SHARPy recycled posterior. Grey: the `β = 1` final population. The plot
range is the prior support; black lines are GWTC-1 published medians (visual
reference only).

In [ ]:
import corner
names  = list(sampler.param_bounds)
post   = np.column_stack([np.asarray(result.posterior_samples[n]) for n in names])
final  = np.column_stack([np.asarray(result.final_samples[n]) for n in names])
ranges = [tuple(map(float, PARAM_BOUNDS[n])) for n in names]

fig = corner.corner(final, labels=names, range=ranges, color="grey",
                    plot_datapoints=False, smooth=1.0, hist_kwargs={"density": True})
corner.corner(post, fig=fig, range=ranges, color="red", plot_datapoints=False,
              smooth=1.0, show_titles=True, title_fmt=".3f",
              truths=[PUBLISHED_MEDIAN[n] for n in names], truth_color="k",
              hist_kwargs={"density": True})
fig.savefig(f"{OUTDIR}/corner_sharpy_mlgw_bbh_GW150914.png", dpi=120)
plt.show()

## 9. Comparison with GWTC-2.1

Overlay against the GWTC-2.1 `C01:Mixed` posterior release for GW150914. The
cell below downloads it from Zenodo (record 6513631) if it isn't already in the
working directory.

In [ ]:
!pip install pesummary

In [ ]:
import os, urllib.request
GWTC_FILE = "IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_cosmo.h5"
if not os.path.exists(GWTC_FILE):
    url = f"https://zenodo.org/records/6513631/files/{GWTC_FILE}"
    print("Downloading", url)
    urllib.request.urlretrieve(url, GWTC_FILE)
print("✓", GWTC_FILE, f"({os.path.getsize(GWTC_FILE)/1e6:.0f} MB)")

In [ ]:
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from pesummary.io import read

# Reload the data into a different variable name because 'data' was overwritten by a numpy array earlier
filename = GWTC_FILE
pe_data = read(filename, disable_prior=True)

# 1. Extract C01:Mixed data from GWTC-2.1
samples_mixed = pe_data.samples_dict['C01:Mixed']

# Extracting and converting relevant parameters
mixed_mc = np.array(samples_mixed['chirp_mass'])
mixed_q = np.array(samples_mixed['mass_ratio'])
mixed_ra = np.mod(np.array(samples_mixed['ra']), 2*np.pi)

# Handle declination/zenith correctly
if 'zenith' in samples_mixed.keys():
    mixed_dec = np.pi/2 - np.array(samples_mixed['zenith'])
else:
    mixed_dec = np.array(samples_mixed['dec'])

mixed_logdl = np.log(np.array(samples_mixed['luminosity_distance']))
mixed_chi_eff = np.array(samples_mixed['chi_eff'])

# Stack the data
mixed_data = np.column_stack([mixed_ra, mixed_dec, mixed_logdl, mixed_mc, mixed_q, mixed_chi_eff])

# 2. Extract MLGW-JAX SEOBNRv5HM data (SHARPy recycled posterior)
v5_ra = np.array(result.posterior_samples['ra'])
v5_dec = np.array(result.posterior_samples['dec'])
v5_logdl = np.log(np.array(result.posterior_samples['distance']))
v5_mc = np.array(result.posterior_samples['mc'])
v5_q = np.array(result.posterior_samples['q'])
v5_chi_1 = np.array(result.posterior_samples['chi_1'])
v5_chi_2 = np.array(result.posterior_samples['chi_2'])

# Calculate chi_eff: (m1*chi_1 + m2*chi_2)/(m1+m2) = (q*chi_1 + chi_2)/(1+q) where q = m2/m1 <= 1
v5_chi_eff = (v5_q * v5_chi_1 + v5_chi_2) / (1.0 + v5_q)

# Stack the MLGW-JAX data (v5_q is already m2/m1 <= 1, no need to invert)
v5_data = np.column_stack([v5_ra, v5_dec, v5_logdl, v5_mc, v5_q, v5_chi_eff])

# 3. Define labels and standardized ranges
labels_comp = [r'$\alpha$', r'$\delta$', r'$\log{D_{L}}$', r'Chirp Mass $[M_{\odot}]$', r'Mass ratio ($q$)', r'$\chi_{eff}$']
ranges_comp = [
    (0, 2*np.pi),        # alpha (RA)
    (-np.pi/2, np.pi/2), # delta (Dec)
    (3.0, 8.5),          # logDL
    (23.0, 38.0),        # Mc
    (0.1, 1.0),          # q
    (-1, 1)              # chi_eff
]

# Set desired font sizes
LABEL_FONT = 16
TITLE_FONT = 18
TICK_FONT = 14
LEGEND_FONT = 20

# 4. Create the Comparison Corner Plot
# MLGW-JAX SEOBNRv5HM (Red) - Base layer
fig = corner.corner(
    v5_data,
    labels=labels_comp,
    range=ranges_comp,
    smooth=1,
    color='red',
    hist_kwargs={'density': True},
    label_kwargs={'fontsize': LABEL_FONT},
    title_kwargs={'fontsize': TITLE_FONT},
    plot_contours=True,
    show_titles=True,
    plot_datapoints=False,
    title_fmt='.3f'
)

# GWTC-2.1 Mixed (Blue) - Overlay
corner.corner(
    mixed_data,
    fig=fig,
    range=ranges_comp,
    smooth=0.9,
    color='blue',
    hist_kwargs={'density': True},
    plot_contours=True,
    fill_contours=False,   # Do not fill contours
    plot_datapoints=False, # Do not plot datapoints
    plot_density=False     # Turn off 2D density color filling
)

# Clean up titles to only show numbers and increase tick font sizes
ndim = len(labels_comp)
axes = np.array(fig.axes).reshape((ndim, ndim))
for i in range(ndim):
    # Extract and modify title on the diagonal
    ax_diag = axes[i, i]
    title = ax_diag.get_title()
    if "=" in title:
        ax_diag.set_title(title.split("=")[-1].strip(), fontsize=TITLE_FONT)

    # Update tick labels size for all axes
    for j in range(ndim):
        axes[i, j].tick_params(axis='both', which='major', labelsize=TICK_FONT)

# 5. Add Legends and Title
proxy_v5 = mlines.Line2D([], [], color='red', label='mlgw_jax_SEOBNRv5HM (SHARPy)')
proxy_mixed = mlines.Line2D([], [], color='blue', label='GWTC-2.1 - Mixed')

fig.legend(handles=[proxy_v5, proxy_mixed], loc='upper right', bbox_to_anchor=(0.95, 0.95), fontsize=LEGEND_FONT, frameon=True, facecolor='#f4f4f4', edgecolor='lightgrey', framealpha=1.0)

# Remove grid lines
for ax in fig.axes:
    ax.grid(False)

plt.savefig(f'{OUTDIR}/corner_fig_no_incl_sharpy.png')
plt.show()

In [ ]:
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from pesummary.io import read

# Reload the data into a different variable name because 'data' was overwritten by a numpy array earlier
filename = GWTC_FILE
pe_data = read(filename, disable_prior=True)

# 1. Extract C01:Mixed data from GWTC-2.1
samples_mixed = pe_data.samples_dict['C01:Mixed']

# Extracting and converting relevant parameters
mixed_mc = np.array(samples_mixed['chirp_mass'])
mixed_q = np.array(samples_mixed['mass_ratio'])
mixed_ra = np.mod(np.array(samples_mixed['ra']), 2*np.pi)

# Handle declination/zenith correctly
if 'zenith' in samples_mixed.keys():
    mixed_dec = np.pi/2 - np.array(samples_mixed['zenith'])
else:
    mixed_dec = np.array(samples_mixed['dec'])

mixed_logdl = np.log(np.array(samples_mixed['luminosity_distance']))
mixed_chi_eff = np.array(samples_mixed['chi_eff'])
mixed_inclination = np.array(samples_mixed['iota']) if 'iota' in samples_mixed.keys() else np.array(samples_mixed['theta_jn'])

# Stack the data
mixed_data = np.column_stack([mixed_ra, mixed_dec, mixed_logdl, mixed_mc, mixed_q, mixed_chi_eff, mixed_inclination])

# 2. Extract MLGW-JAX SEOBNRv5HM data (SHARPy recycled posterior)
v5_ra = np.array(result.posterior_samples['ra'])
v5_dec = np.array(result.posterior_samples['dec'])
v5_logdl = np.log(np.array(result.posterior_samples['distance']))
v5_mc = np.array(result.posterior_samples['mc'])
v5_q = np.array(result.posterior_samples['q'])
v5_chi_1 = np.array(result.posterior_samples['chi_1'])
v5_chi_2 = np.array(result.posterior_samples['chi_2'])
v5_inclination = np.array(result.posterior_samples['inclination'])

# Calculate chi_eff: (m1*chi_1 + m2*chi_2)/(m1+m2) = (q*chi_1 + chi_2)/(1+q) where q = m2/m1 <= 1
v5_chi_eff = (v5_q * v5_chi_1 + v5_chi_2) / (1.0 + v5_q)

# Stack the MLGW-JAX data (v5_q is already m2/m1 <= 1, no need to invert)
v5_data = np.column_stack([v5_ra, v5_dec, v5_logdl, v5_mc, v5_q, v5_chi_eff, v5_inclination])

# 3. Define labels and standardized ranges
labels_comp = [r'$\alpha$', r'$\delta$', r'$\log{D_{L}}$', r'Chirp Mass $[M_{\odot}]$', r'Mass ratio ($q$)', r'$\chi_{eff}$', r'$\iota$']
ranges_comp = [
    (0, 2*np.pi),        # alpha (RA)
    (-np.pi/2, np.pi/2), # delta (Dec)
    (3.0, 8.5),          # logDL
    (23.0, 38.0),        # Mc
    (0.1, 1.0),          # q
    (-1, 1),             # chi_eff
    (0, np.pi)           # inclination
]

# Set desired font sizes
LABEL_FONT = 16
TITLE_FONT = 18
TICK_FONT = 14
LEGEND_FONT = 20

# 4. Create the Comparison Corner Plot
# MLGW-JAX SEOBNRv5HM (Red) - Base layer
fig = corner.corner(
    v5_data,
    labels=labels_comp,
    range=ranges_comp,
    smooth=1,
    color='red',
    hist_kwargs={'density': True},
    label_kwargs={'fontsize': LABEL_FONT},
    title_kwargs={'fontsize': TITLE_FONT},
    plot_contours=True,
    show_titles=True,
    plot_datapoints=False,
    title_fmt='.3f'
)

# GWTC-2.1 Mixed (Blue) - Overlay
corner.corner(
    mixed_data,
    fig=fig,
    range=ranges_comp,
    smooth=0.9,
    color='blue',
    hist_kwargs={'density': True},
    plot_contours=True,
    fill_contours=False,   # Do not fill contours
    plot_datapoints=False, # Do not plot datapoints
    plot_density=False     # Turn off 2D density color filling
)

# Clean up titles to only show numbers and increase tick font sizes
ndim = len(labels_comp)
axes = np.array(fig.axes).reshape((ndim, ndim))
for i in range(ndim):
    # Extract and modify title on the diagonal
    ax_diag = axes[i, i]
    title = ax_diag.get_title()
    if "=" in title:
        ax_diag.set_title(title.split("=")[-1].strip(), fontsize=TITLE_FONT)

    # Update tick labels size for all axes
    for j in range(ndim):
        axes[i, j].tick_params(axis='both', which='major', labelsize=TICK_FONT)

# 5. Add Legends and Title
proxy_v5 = mlines.Line2D([], [], color='red', label='mlgw_jax_SEOBNRv5HM (SHARPy)')
proxy_mixed = mlines.Line2D([], [], color='blue', label='GWTC-2.1 - Mixed')

fig.legend(handles=[proxy_v5, proxy_mixed], loc='upper right', bbox_to_anchor=(0.95, 0.95), fontsize=LEGEND_FONT, frameon=True, facecolor='#f4f4f4', edgecolor='lightgrey', framealpha=1.0)

# Remove grid lines
for ax in fig.axes:
    ax.grid(False)

plt.savefig(f'{OUTDIR}/corner_fig_with_incl_sharpy.png')
plt.show()

In [ ]:
import pandas as pd

# Convert the posterior samples dictionary to a pandas DataFrame
df_samples = pd.DataFrame(result.posterior_samples)

# Save the DataFrame to a text file (tab-separated)
filename = 'posterior_samples_mlgw_bbh_sharpy.txt'
df_samples.to_csv(filename, sep='\t', index=False)
print(f'Saved posterior samples to {filename}')

# Download the file to your local machine
if RUNNING_ON_COLAB:
    from google.colab import files
    files.download(filename)

## 10. Save everything to Google Drive

Creates a **new folder for every run** under `MyDrive/GW150914_SHARPy_results/`,
named with the date/time, particle count and metric, e.g.
`run_20261015_143210_N4000_hessian/`. It contains:

| File | Content |
|------|---------|
| `posterior_samples_recycled.txt` | recycled posterior draws (tab-separated, one column per parameter) |
| `final_population_beta1.txt` | the β = 1 particle population |
| `run_summary.json` | logZ ± err, number of steps, wall-clock, sampler settings, prior bounds/shapes |
| `smc_diagnostics.npz` | per-step β, ESS, NUTS acceptance, divergent fraction |
| `smc_history.npz` | every tempered population (unit cube) + its log-likelihood, so you can redo the recycling later |
| `*.png`, `*_result.npz` | everything already written to `OUTDIR` (corner plots, sampler result file) |

Outside Colab it writes the same folder under the local working directory.

In [ ]:
import os, json, shutil, datetime
import pandas as pd

if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/GW150914_SHARPy_results"
else:
    DRIVE_ROOT = "GW150914_SHARPy_results"

stamp    = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = os.path.join(DRIVE_ROOT, f"run_{stamp}_N{NUM_PARTICLES}_{METRIC}")
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Posterior samples (recycled) and final beta = 1 population, physical units.
pd.DataFrame(result.posterior_samples).to_csv(
    f"{SAVE_DIR}/posterior_samples_recycled.txt", sep="\t", index=False)
pd.DataFrame(result.final_samples).to_csv(
    f"{SAVE_DIR}/final_population_beta1.txt", sep="\t", index=False)

# 2. Run summary + settings.
summary = dict(
    event="GW150914", waveform="mlgw_bbh_jax SEOBNRv5HM model_4 (2,2)",
    sampler="SHARPy (gwgpu_jax.GWgpu_jaxSHARPySampler)",
    logZ=result.logZ, logZ_err=result.logZ_err, n_steps=result.n_steps,
    wall_clock_s=float(elapsed),
    n_posterior_recycled=len(next(iter(result.posterior_samples.values()))),
    settings=dict(num_particles=NUM_PARTICLES, alpha=ALPHA, step_size=STEP_SIZE,
                  max_num_doublings=MAX_NUM_DOUBLINGS, num_mcmc_steps=NUM_MCMC_STEPS,
                  metric=METRIC, chunk_size=CHUNK_SIZE, rng_seed=43),
    param_bounds={k: [float(a), float(b)] for k, (a, b) in PARAM_BOUNDS.items()},
    priors=PRIORS, periodic_params=list(sampler.periodic_params),
    data=dict(duration=DURATION, sampling_rate=SAMPLING_RATE, f_min=F_MIN, f_max=F_MAX),
)
with open(f"{SAVE_DIR}/run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# 3. SMC diagnostics and the full tempered history.
np.savez(f"{SAVE_DIR}/smc_diagnostics.npz", betas=result.betas, ess=result.ess,
         acceptance=result.acceptance, divergences=result.divergences)
np.savez_compressed(
    f"{SAVE_DIR}/smc_history.npz",
    param_names=np.array(sampler.param_names),
    particles_unit_cube=np.stack(result.history["particles"]),   # (n_steps, N, d)
    logL=np.stack(result.history["logL"]),                         # (n_steps, N)
    log_weights=np.stack(result.history["log_weights"]),           # (n_steps, N)
    betas=np.asarray(result.history["beta"]),
)

# 4. Copy whatever is already in OUTDIR (sampler .npz, corner plots).
if os.path.isdir(OUTDIR):
    for fn in os.listdir(OUTDIR):
        src = os.path.join(OUTDIR, fn)
        if os.path.isfile(src):
            shutil.copy2(src, SAVE_DIR)

print(f"✓ saved to {SAVE_DIR}")
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f"   {fn:45s} {os.path.getsize(os.path.join(SAVE_DIR, fn))/1e6:8.2f} MB")